In [ ]:
import pennylane as qml
from opt_pipeline import *
import numpy as np

In [2]:
[ds] = qml.data.load("ketgpt")

In [16]:
f = []
for seed in range(1000):
    @qml.qnode(qml.device('default.qubit'))
    def circuit():
        for op in ds.circuits[seed]:
            name = op.name
            params = op.parameters
            wires = op.wires
            if name == 'QubitUnitary':
                continue
            elif name == 'CZ':
                qml.Hadamard(wires[1])
                qml.CNOT(wires)
                qml.Hadamard(wires[1])
            elif name == 'U1':
                qml.RZ(params[0], wires=wires)
            elif name == 'U2':
                qml.RZ(params[0], wires=wires)
                qml.RY(np.pi/2, wires=wires)
                qml.RZ(params[1], wires=wires)
            else:
                qml.apply(op)
        return qml.state()
    
    num_q = qml.specs(circuit)()['resources'].num_wires
    if num_q<9:
        try:
            U1 = qml.matrix(circuit, wire_order=range(num_q))()
        except: # when the wire_order is not consecutive order
            continue
        circuit_info = optimizeCircuit_info(circuit) # processed by the proposed optimizer
        qnode = info_to_qnode(circuit_info)
        num_q_2 = qml.specs(circuit)()['resources'].num_wires
        U2 = qml.matrix(circuit, wire_order=range(num_q))()

        if num_q==num_q_2:
            d = 2**num_q
            fidelity = abs(np.trace(np.conj(U1.T) @ U2)) / d
            f.append(fidelity)
            print(f"Process fidelity: {np.round(fidelity,5)} at seed={seed}")

Process fidelity: 1.0 at seed=11
Process fidelity: 1.0 at seed=16
Process fidelity: 1.0 at seed=17
Process fidelity: 1.0 at seed=22
Process fidelity: 1.0 at seed=27
Process fidelity: 1.0 at seed=44
Process fidelity: 1.0 at seed=48
Process fidelity: 1.0 at seed=55
Process fidelity: 1.0 at seed=63
Process fidelity: 1.0 at seed=72
Process fidelity: 1.0 at seed=75
Process fidelity: 1.0 at seed=83
Process fidelity: 1.0 at seed=100
Process fidelity: 1.0 at seed=104
Process fidelity: 1.0 at seed=106
Process fidelity: 1.0 at seed=127
Process fidelity: 1.0 at seed=144
Process fidelity: 1.0 at seed=151
Process fidelity: 1.0 at seed=160
Process fidelity: 1.0 at seed=163
Process fidelity: 1.0 at seed=165
Process fidelity: 1.0 at seed=180
Process fidelity: 1.0 at seed=206
Process fidelity: 1.0 at seed=221
Process fidelity: 1.0 at seed=253
Process fidelity: 1.0 at seed=268
Process fidelity: 1.0 at seed=269
Process fidelity: 1.0 at seed=274
Process fidelity: 1.0 at seed=287
Process fidelity: 1.0 at s

In [18]:
flag = np.all(np.isclose(f, 1.0))
if flag:
    print(f'The process fidelity is one at all the {len(f)} seeds')

The process fidelity is one at all the 79 seeds
